<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/eval/STEP7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly Segmentation (Step 7)

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eval

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eval


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')

In [4]:
#!pip install ood-metrics

In [5]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

In [6]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


In [28]:
parser = ArgumentParser()
parser.add_argument(
    "--input",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png",
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)
parser.add_argument('--loadDir',default="../trained_models/")
parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
parser.add_argument('--loadModel', default="erfnet.py")
parser.add_argument('--subset', default="val")  #can be val or train (must have labels)
#parser.add_argument('--datadir', default="???????")
parser.add_argument('--num-workers', type=int, default=4)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
args = parser.parse_args(args=[])
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
ood_gts_list = []

if not os.path.exists('results.txt'):
    open('results.txt', 'w').close()
file = open('results.txt', 'a')

modelpath = args.loadDir + args.loadModel
weightspath = args.loadDir + args.loadWeights

print ("Loading model: " + modelpath)
print ("Loading weights: " + weightspath)

model = ERFNet(NUM_CLASSES)

if (not args.cpu):
    model = torch.nn.DataParallel(model).cuda()

def load_my_state_dict(model, state_dict):  #custom function to load model when not all dict elements
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
            else:
                print(name, " not loaded")
                continue
        else:
            own_state[name].copy_(param)
    return model

model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
print ("Model and weights LOADED successfully")
model.eval()

Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully


DataParallel(
  (module): ERFNet(
    (encoder): Encoder(
      (initial_block): DownsamplerBlock(
        (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layers): ModuleList(
        (0): DownsamplerBlock(
          (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1-5): 5 x non_bottleneck_1d(
          (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
          (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
          (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True

In [36]:
#for path in glob.glob(os.path.expanduser(str(args.input[0]))):

# Controlliamo se args.input è una lista o una stringa
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

for path in glob.glob(os.path.expanduser(str(input_pattern))):
    print(path)
    images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
    #images = images.permute(0,3,1,2)
    with torch.no_grad():
        result = model(images)

    #compute max logit
    anomaly_result_maxlogit = 1.0 - np.max(result.squeeze(0).data.cpu().numpy(), axis=0)

    #compute MSP
    probs = torch.nn.functional.softmax(result, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    #compute Max Entropy
    probs = torch.nn.functional.softmax(result, dim=1).squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "FS_LostFound_full" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
      #in RoadAnomaly 2 indica anomalia --> viene trasformato in 1 per uniformare
        ood_gts = np.where((ood_gts==2), 1, ood_gts)
    #if "LostFound" in pathGT:
    #    ood_gts = np.where((ood_gts==0), 255, ood_gts)
    #    ood_gts = np.where((ood_gts==1), 0, ood_gts)
    #    ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

    #if "Streethazard" in pathGT:
     #   ood_gts = np.where((ood_gts==14), 255, ood_gts)
      #  ood_gts = np.where((ood_gts<20), 0, ood_gts)
       # ood_gts = np.where((ood_gts==255), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        #print("no anomalie")
        continue   #se non c'è nessuna anomalia, allora salta l'immagine e passa a valutare la successiva
    else:
          print('anomalia')
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)

    del result, anomaly_result_maxlogit,anomaly_result_msp, anomaly_result_maxentropy, ood_gts, mask
    torch.cuda.empty_cache()

file.write( "------\n")

/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/48.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/63.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/88.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/89.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/77.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/71.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/59.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Vali

7

In [37]:
ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')

file.write(('    AUPRC score maxlogit:' + str(prc_auc*100.0) + '   FPR@TPR95 maxlogit:' + str(fpr*100.0) ))

#MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')

file.write(('    AUPRC score msp:' + str(prc_auc*100.0) + '   FPR@TPR95 msp:' + str(fpr*100.0) ))

#max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')

file.write(('    AUPRC score maxentropy:' + str(prc_auc*100.0) + '   FPR@TPR95 maxentropy:' + str(fpr*100.0) ))

file.close()

AUPRC score maxlogit: 3.3014401838550618
FPR@TPR95 maxlogit: 45.494847365050845
AUPRC score msp: 1.749039030843141
FPR@TPR95 msp: 50.594030268678914
AUPRC score maxentropy: 2.5839564298873796
FPR@TPR95 maxentropy: 50.162776109239616
